# EDA — Home Credit Default Risk

Análise exploratória inicial da base de referência para entender estrutura, qualidade dos dados e distribuição da variável-alvo `TARGET`.

## 1. Preparação do ambiente

As dependências são instaladas pela imagem Docker a partir dos arquivos `requirements*.txt`. Os dados são baixados uma vez pelo Kaggle CLI para `data/raw`; este notebook não autentica nem baixa dados.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")

## 2. Download e inventário dos dados

Aceite as regras da competição no site do Kaggle e, no terminal do Mac, execute uma vez a partir da raiz do projeto:

```bash
kaggle auth login
mkdir -p data/raw
kaggle competitions download home-credit-default-risk --path data/raw
unzip -o data/raw/home-credit-default-risk.zip -d data/raw
```

Os CSVs permanecem em `data/raw` e ficam disponíveis no container pelo volume do projeto.

In [ ]:
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

data_path = project_root / "data" / "raw"
if not data_path.is_dir():
    raise FileNotFoundError(
        f"Pasta de dados não encontrada: {data_path}. Baixe os arquivos "
        "conforme as instruções acima."
    )

csv_files = sorted(data_path.rglob("*.csv"))
if not csv_files:
    raise FileNotFoundError(
        f"Nenhum CSV encontrado em {data_path}. Baixe e extraia "
        "o arquivo da competição."
    )

print(f"Arquivos CSV encontrados: {len(csv_files)} em {data_path}")

In [ ]:
file_inventory = pd.DataFrame(
    {
        "arquivo": [file.name for file in csv_files],
        "tamanho_mb": [round(file.stat().st_size / 1024**2, 2) for file in csv_files],
    }
)
file_inventory

## 3. Base principal de treinamento

A análise começa por `application_train.csv`, que contém as características das solicitações e o desfecho binário `TARGET`.

In [ ]:
train_file = next(
    (file for file in csv_files if file.name == "application_train.csv"), None
)
if train_file is None:
    raise FileNotFoundError("application_train.csv não foi encontrado em data/raw.")

df = pd.read_csv(train_file)
required_columns = {"SK_ID_CURR", "TARGET"}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Colunas obrigatórias ausentes: {sorted(missing_columns)}")

print(f"Linhas: {df.shape[0]:,}")
print(f"Colunas: {df.shape[1]:,}")
df.head()

In [ ]:
type_summary = (
    df.dtypes.astype(str)
    .value_counts()
    .rename_axis("tipo")
    .reset_index(name="quantidade_colunas")
)
type_summary

## 4. Qualidade dos dados

In [ ]:
duplicate_rows = int(df.duplicated().sum())
duplicate_ids = int(df["SK_ID_CURR"].duplicated().sum())

pd.Series(
    {
        "linhas_duplicadas": duplicate_rows,
        "ids_duplicados": duplicate_ids,
        "total_nulos": int(df.isna().sum().sum()),
    },
    name="resultado",
)

In [ ]:
missing = (
    df.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("percentual_nulos")
    .to_frame()
)
missing.head(20)

In [ ]:
missing.head(20).sort_values("percentual_nulos").plot.barh(
    figsize=(9, 7), legend=False, title="20 colunas com maior percentual de nulos"
)
plt.xlabel("Valores ausentes (%)")
plt.ylabel("Coluna")
plt.tight_layout()
plt.show()

## 5. Distribuição da variável-alvo

No conjunto de treinamento, `TARGET = 1` representa cliente com dificuldade de pagamento e `TARGET = 0` representa os demais casos.

In [ ]:
target_distribution = (
    df["TARGET"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("TARGET")
    .to_frame("quantidade")
)
target_distribution["percentual"] = (
    target_distribution["quantidade"] / len(df) * 100
).round(2)
target_distribution

In [ ]:
ax = sns.countplot(data=df, x="TARGET", hue="TARGET", legend=False)
ax.set(title="Distribuição da variável-alvo", xlabel="TARGET", ylabel="Quantidade")
plt.tight_layout()
plt.show()

## 6. Visão estatística inicial

In [ ]:
df.select_dtypes(include="number").describe().T

In [ ]:
categorical_summary = pd.DataFrame(
    {
        "valores_unicos": df.select_dtypes(exclude="number").nunique(dropna=False),
        "percentual_nulos": df.select_dtypes(exclude="number")
        .isna()
        .mean()
        .mul(100)
        .round(2),
    }
).sort_values("valores_unicos", ascending=False)
categorical_summary

## 7. Próximos passos

- Definir as variáveis da primeira versão do modelo.
- Formalizar o contrato de dados com regras para esquema, domínio e valores ausentes.
- Separar a base de referência das bases que serão usadas na simulação de drift.
- Estabelecer métricas adequadas ao desbalanceamento da variável-alvo.